# Causal Self-Attention (Masked Self-Attention)
## Chapter 3: Autoregressive Masking & Attention Dropout

In standard self-attention, each token can attend to all positions (both past and future). However, **generative Large Language Models** (such as GPT) must be **autoregressive**: when predicting the next token $t_i$, the model must only attend to past and current tokens ($t_0, \dots, t_i$), without "peeking" into future tokens ($t_{i+1}, \dots, t_{T-1}$).

---

### The Mechanism of Causal Masking:
1. Compute raw attention scores:
   $$\mathbf{S} = \frac{\mathbf{Q} \mathbf{K}^\top}{\sqrt{d_k}}$$

2. Construct a **lower triangular boolean mask** $\mathbf{M}$:
   $$\mathbf{M}_{i,j} = \begin{cases} 1 & \text{if } i \ge j \text{ (allow: past and self)} \\ 0 & \text{if } i < j \text{ (block: future tokens)} \end{cases}$$

3. Replace future positions ($M_{i,j} = 0$) with $-\infty$:
   $$\mathbf{S}_{\text{masked}} = \text{masked\_fill}(\mathbf{S}, \neg\mathbf{M}, -\infty)$$

4. Applying $\text{softmax}(\mathbf{S}_{\text{masked}})$ maps $-\infty \rightarrow e^{-\infty} = 0$:
   $$\text{Row } i: [a_{i,0}, a_{i,1}, \dots, a_{i,i}, 0, 0, \dots, 0]$$
   Future tokens receive exactly **0% attention probability**.

5. **Attention Dropout**: Apply dropout to the normalized attention matrix during training to prevent co-adaptation of attention heads.

---
## 1. Imports and Dependencies

In [1]:
import torch 
import math 
import torch.nn as nn 
import torch.nn.functional as F

---
## 2. Implementing `CausalAttention`

Here we implement `CausalAttention`:
- Computes Queries, Keys, and Values.
- Constructs the causal mask using `torch.tril` (lower-triangular matrix).
- Applies `masked_fill` with `-inf` to future tokens.
- Normalizes scores via softmax and applies dropout regularization (`F.dropout`).
- Aggregates Values $\mathbf{V}$ using masked attention weights.

In [9]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)

        d_k = key.size(-1)
        atten_score = query @ key.transpose(-2, -1)

        seq_len = atten_score.size(-1)

        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()

        masked_atten_score = atten_score.masked_fill(~mask, float('-inf'))

        atten_weights = torch.softmax(masked_atten_score / math.sqrt(d_k), dim=-1)

        torch.manual_seed(789)
        drop_atten_weights = F.dropout(atten_weights, p=0.1)

        context = drop_atten_weights @ value
        return context

---
## 3. Verification & Execution

Let's test `CausalAttention` on sample input:
- Input shape: $(B=1, T=3, d_{in}=4)$
- Output projection: $d_{out}=2$
- Each token $t_i$ computes its output using only representations up to position $i$.

In [10]:
d_in = 4
d_out = 2
torch.manual_seed(789)

inputs = torch.randn(1, 3, d_in)

sa = CausalAttention(d_in, d_out)
print(sa(inputs))

tensor([[[-0.4684,  0.0188],
         [-0.3012, -0.2176],
         [-0.1737, -0.1098]]], grad_fn=<UnsafeViewBackward0>)
